# EMI Simulation using FEniCSx

This notebook implements an **Electromagnetic Induction (EMI)** simulation following the primal single-domain formulation described in the [FEniCS-in-the-wild tutorial](https://scientificcomputing.github.io/fenics-in-the-wild/src/ucs/emi/emi_primal_single.html).

## Environment setup
Follow the Fenicsx installation setup (https://fenicsproject.org/download/). I.e. in the terminal (and later choose this env as the kernel for this notebook).
```bash
conda create -n fenicsx-env
conda activate fenicsx-env
conda install -c conda-forge fenics-dolfinx mpich pyvista
```

Install some additional libraries
```bash
conda install ipykernel pip
pip install scifem
```


## Physical Background
The goal is to solve for the electrical potentials in the intracellular ($\Omega_i$) and extracellular ($\Omega_e$) spaces. The two domains are coupled at the cell membrane ($\Gamma$) by a capacitive term, representing the membrane potential.

The governing equations are typically diffusion equations in each domain, coupled by a jump condition at the interface:
$$\sigma_e \nabla u_e \cdot n_e = \sigma_i \nabla u_i \cdot n_i = C_m \frac{\partial (u_i - u_e)}{\partial t} + I_{ion}$$

In this simulation, we solve the steady-state (or quasi-static) problem using a mixed formulation.

## 1. Setup and Imports

We begin by importing the necessary libraries. We use `dolfinx` for the finite element method, `ufl` for the variational forms, and `scifem` for advanced mesh operations like submesh extraction and interface identification.

In [ ]:
from mpi4py import MPI
from petsc4py import PETSc
import dolfinx
import numpy as np
from ufl import (
    inner, grad, TestFunctions, TrialFunctions, FacetNormal, 
    MixedFunctionSpace, Measure, SpatialCoordinate, div
)
import scifem
from pathlib import Path

In [ ]:
# Path to the results folder with meshes/mesh.xdmf
CELL_PATH = Path("../emimesh/results/cells/oligos/oligo_1")

## 2. Mesh Loading and Domain Decomposition

We load the mesh and the associated cell tags from the XDMF file. According to the dataset mapping:
- **0**: Extracellular space ($\Omega_e$)
- **2**: Intracellular space ($\Omega_i$)

We use `scifem.extract_submesh` to create separate meshes for each domain, which is required for the mixed function space formulation.

In [ ]:
with dolfinx.io.XDMFFile(MPI.COMM_WORLD, CELL_PATH / "meshes/mesh.xdmf", "r") as xdmf:
    omega = xdmf.read_mesh(name="Grid")
    cell_tags = xdmf.read_meshtags(mesh=omega, name="Grid")

interior_marker = 2
exterior_marker = 0

# Extract submeshes for intracellular and extracellular domains
omega_i, interior_to_parent, _, _, _ = scifem.extract_submesh(
    omega, cell_tags, interior_marker
)
omega_e, exterior_to_parent, e_vertex_to_parent, _, _ = scifem.extract_submesh(
    omega, cell_tags, exterior_marker
)

# Identify the interface Gamma between the two domains
gamma_facets = scifem.find_interface(cell_tags, interior_marker, exterior_marker)

print(f"Mesh loaded successfully. Interior cells: {omega_i.topology.index_map(omega_i.topology.dim).size_local}")
print(f"Exterior cells: {omega_e.topology.index_map(omega_e.topology.dim).size_local}")

## 3. Variational Formulation

We define the function spaces for each domain and combine them into a mixed space $W = V_i \times V_e$. 

The bilinear form $a(u, v)$ consists of:
1. Diffusion in the intracellular space $\int_{\Omega_i} \sigma_i \nabla u_i \cdot \nabla v_i \, dx$
2. Diffusion in the extracellular space $\int_{\Omega_e} \sigma_e \nabla u_e \cdot \nabla v_e \, dx$
3. Interface coupling terms that enforce the capacitive jump condition.

In [ ]:
# Physics parameters
sigma_e = dolfinx.fem.Constant(omega_e, 2.0)
sigma_i = dolfinx.fem.Constant(omega_i, 1.0)
Cm = dolfinx.fem.Constant(omega, 1.0)
dt = dolfinx.fem.Constant(omega, 1.0e-2)
T = Cm / dt

element = ("Lagrange", 1)
Vi = dolfinx.fem.functionspace(omega_i, element)
Ve = dolfinx.fem.functionspace(omega_e, element)
W = MixedFunctionSpace(Vi, Ve)
vi, ve = TestFunctions(W)
ui, ue = TrialFunctions(W)

# Interface data for consistent restrictions
ordered_integration_data = scifem.compute_interface_data(cell_tags, gamma_facets)
interface_tag = 100
dGamma = Measure(
    "dS",
    domain=omega,
    subdomain_data=[(interface_tag, ordered_integration_data.flatten())],
    subdomain_id=interface_tag,
)

# We define a convention for restrictions to the interface
i_res = "+" if interior_marker < exterior_marker else "-"
e_res = "-" if interior_marker < exterior_marker else "+"
tr_ui = ui(i_res)
tr_ue = ue(e_res)
tr_vi = vi(i_res)
tr_ve = ve(e_res)

# Bilinear form
a = sigma_e * inner(grad(ue), grad(ve)) * Measure("dx", domain=omega_e)
a += sigma_i * inner(grad(ui), grad(vi)) * Measure("dx", domain=omega_i)
a += T * (tr_ue - tr_ui) * tr_ve * dGamma
a += T * (tr_ui - tr_ue) * tr_vi * dGamma

# Linear form (source terms)
L = 0.0 # In a real scenario, add f_i, f_e or interface currents here
L += T * 0.0 * (tr_vi - tr_ve) * dGamma # Example source

print("Variational form defined.")

## 4. Boundary Conditions

We apply a Dirichlet boundary condition to the outer boundary of the extracellular domain $\Omega_e$. This typically represents the ground or a fixed potential at the edge of the ROI.

In [ ]:
# Transfer parent mesh tags to submesh to identify boundary
sub_tag, _ = scifem.transfer_meshtags_to_submesh(
    cell_tags, omega_e, e_vertex_to_parent, exterior_to_parent
)

omega_e.topology.create_connectivity(omega_e.topology.dim - 1, omega_e.topology.dim)
exterior_facets = dolfinx.mesh.exterior_facet_indices(omega_e.topology)
bc_dofs = dolfinx.fem.locate_dofs_topological(
    Ve, omega_e.topology.dim - 1, exterior_facets
)

u_bc = dolfinx.fem.Function(Ve)
u_bc.interpolate(lambda x: np.zeros(x.shape[1]))
bc = dolfinx.fem.dirichletbc(u_bc, bc_dofs)

print("Boundary conditions applied.")

## 5. Solver and Execution
We use the `LinearProblem` class from `dolfinx.fem.petsc` to solve the system. We apply a block-preconditioner to handle the coupling between the interior and exterior potentials.

In [ ]:
from dolfinx.fem.petsc import LinearProblem
from ufl import extract_blocks

# Setup the problem
ui_sol = dolfinx.fem.Function(Vi, name="ui")
ue_sol = dolfinx.fem.Function(Ve, name="ue")
entity_maps = [interior_to_parent, exterior_to_parent]

petsc_options = {
    "ksp_type": "cg",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
    "ksp_rtol": 1e-12,
    "ksp_atol": 1e-12,
}

problem = LinearProblem(
    extract_blocks(a),
    extract_blocks(L),
    u=[ui_sol, ue_sol],
    bcs=[bc],
    petsc_options=petsc_options,
    entity_maps=entity_maps,
)

problem.solve()

print(f"Simulation converged. Solver iterations: {problem.solver.getIterationNumber()}")

## 6. Results
The potentials $u_i$ and $u_e$ can now be exported or visualized using PyVista or ParaView.

In [ ]:
with dolfinx.io.XDMFFile(MPI.COMM_WORLD, "emi_result.xdmf", "w") as xdmf:
    xdmf.write_mesh(omega_i)
    xdmf.write_function(ui_sol)

print("Interior potential saved to emi_result.xdmf")